Which parameter values best account for the observed recall sequences? Fit a model to the selected trials, or open a saved fit and score its estimates on the recorded trials.

<div id="notebook-reference">

| Parameter | Meaning |
| --- | --- |
| `fit_action` | `"inspect"` opens the named records. `"estimate"` runs the optimizer. For a batch that can resume, `"fit_if_missing"` opens each existing output and estimates the missing fits. |
| `saved_fit_paths` | Model names and input JSON paths for inspection. The record supplies its model, parameter values and trial selection. |
| `data_path`, `trial_query` | Observation file and trial selection for estimation. A saved record supplies its own observation path when available. |
| `pooled`, `max_subjects`, `subject_indices` | Estimate one set of parameters across the selection or one per participant. Zero for `max_subjects` keeps all IDs. Indices select from those IDs in sorted order. |
| `model_configs` | One entry per model to estimate. Specify its factory, component choices, fixed values, bounds for free parameters and search seed. Optional `embedding_path` and `emotion_feature_path` supply vocabulary features. `feature_column` selects the categorical column; `concat_features=True` places it before the embeddings. |
| `loss_fn_path`, `fit_alg_path`, `fit_hyperparams` | The objective to minimize, the optimizer and its search settings. `enable_x64` controls numerical precision. |
| `project_root`, `output_dir`, `fit_dir` | Resolve relative paths under the project. New records go to `fits/` under the output directory unless you choose a separate fit directory. |
| `data_tag`, `base_run_tag` | Parts of the output filename. A model's optional `output_stem` supplies its own name. |

Each model produces a table of estimates and objective scores and a path to its fit JSON. Estimating writes that record. Inspecting reads it. New records carry the model’s `simulation_function` when supplied, defaulting to the forced-stop trial simulator. Participant jobs append their selected indices to the filename, for example `_sub0.json`.

</div>

## Configure this run

In [1]:
#| code-summary: "Saved fits, observations and model settings"

from typing import Literal

type FitAction = Literal["inspect", "estimate", "fit_if_missing"]

fit_action: FitAction = "inspect"
saved_fit_paths = {"CMR": "data/fits/Dupertuys2026_CMR_demo.json"}

# Settings for a new estimate.
data_path = "data/Dupertuys2026.h5"
trial_query = None
pooled = True
max_subjects = 0
subject_indices = []

epsilon = 2**-52
model_configs = {
    "CMR": {
        "label": "CMR",
        "model_factory": "jaxcmr.models.cmr.make_factory",
        "components": {
            "mfc_create_fn": "jaxcmr.components.linear_memory.init_mfc",
            "mcf_create_fn": "jaxcmr.components.linear_memory.init_mcf",
            "context_create_fn": "jaxcmr.components.context.init",
            "response_policy_create_fn": "jaxcmr.components.response.RecallUntilExhausted",
        },
        "parameters": {
            "fixed": {
                "learn_after_context_update": False,
                "allow_repeated_recalls": False,
            },
            "free": {
                "encoding_drift_rate": [epsilon, 1 - epsilon],
                "start_drift_rate": [epsilon, 1 - epsilon],
                "recall_drift_rate": [epsilon, 1 - epsilon],
                "learning_rate": [epsilon, 1 - epsilon],
                "shared_support": [epsilon, 100.0],
                "item_support": [epsilon, 100.0],
                "primacy_scale": [epsilon, 100.0],
                "primacy_decay": [epsilon, 100.0],
                "choice_sensitivity": [epsilon, 100.0],
            },
        },
        "seed": 1,
    },
}

loss_fn_path = (
    "jaxcmr.loss.sequence_likelihood.ExcludeTerminationLikelihoodLoss"
)
fit_alg_path = "jaxcmr.fitting.ScipyDE"
fit_hyperparams = {
    "num_steps": 1000,
    "pop_size": 15,
    "best_of": 3,
    "relative_tolerance": 0.0001,
    "absolute_tolerance": 0.0,
    "cross_over_rate": 0.9,
    "diff_w": [0.5, 1.0],
    "init": "latinhypercube",
    "polish": False,
    "display_iterations": False,
    "progress_bar": False,
}

enable_x64 = True
project_root = None
output_dir = "results"
fit_dir = None
data_tag = None
base_run_tag = "demo"

## Load data and select trials

In a participant job, `subject_indices=[0]` selects the first participant in sorted ID order. The fit record keeps that participant’s actual ID and the rows selected before and after applying the index. Saved fits use their recorded trial rows. Older records can supply a trial query in their place.

In [2]:
#| code-summary: "Install JAXCMR in Colab"

import sys

if "google.colab" in sys.modules:
    %pip install --upgrade "jaxcmr @ git+https://github.com/githubpsyche/jaxcmr.git@main"

In [3]:
#| code-summary: "Import fitting and dataset operations"

import json

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from IPython.display import display

import jaxcmr
from jaxcmr.dataset import generate_trial_mask
from jaxcmr.examples import copy_example_inputs
from jaxcmr.fitting import make_scipy_loss_fn
from jaxcmr.helpers import import_from_string, load_data, resolve_project_path, resolve_project_root
from jaxcmr.models import model_factory_from_paths

jax.config.update("jax_enable_x64", enable_x64)
root = resolve_project_root(project_root)

In [4]:
#| code-summary: "Locate saved fits and new fit destinations"

if fit_action == "inspect":
    copy_example_inputs(saved_fit_paths.values(), base_dir=root)
    fit_paths = {name: resolve_project_path(path, root) for name, path in saved_fit_paths.items()}
elif fit_action in ("estimate", "fit_if_missing"):
    directory = resolve_project_path(fit_dir, root) if fit_dir else resolve_project_path(output_dir, root) / "fits"
    tag = data_tag or resolve_project_path(data_path, root).stem
    run_tag = f"{base_run_tag}_best_of_{fit_hyperparams.get('best_of', 1)}"
    if max_subjects:
        run_tag += f"_nsubs_{max_subjects}"
    suffix = "_sub" + "_".join(str(i) for i in subject_indices) if subject_indices else ""
    fit_paths = {
        name: directory / (spec.get("output_stem", f"{tag}_{name}_{run_tag}") + suffix + ".json")
        for name, spec in model_configs.items()
    }
    if len(set(fit_paths.values())) != len(fit_paths):
        raise ValueError("Choose a distinct output filename for each model.")
else:
    raise ValueError(f"Unknown fit action: {fit_action!r}")

if not fit_paths:
    raise ValueError("Supply at least one model to fit or inspect.")

records = {
    name: json.loads(path.read_text()) for name, path in fit_paths.items()
    if fit_action == "inspect" or (fit_action == "fit_if_missing" and path.is_file())
}
models_to_fit = [name for name in fit_paths if name not in records]

Reusing matching input: data/fits/Dupertuys2026_CMR_demo.json


In [5]:
#| code-summary: "Load the observation files"

observation_paths = {name: records.get(name, {}).get("data_path", data_path) for name in fit_paths}
copy_example_inputs(observation_paths.values(), base_dir=root)
datasets = {path: load_data(resolve_project_path(path, root)) for path in set(observation_paths.values())}

Reusing matching input: data/Dupertuys2026.h5


In [6]:
#| code-summary: "Select trials for new estimates"

if models_to_fit:
    data = datasets[data_path]
    subjects = np.asarray(data["subject"]).ravel()
    participant_ids = np.unique(subjects)
    if max_subjects:
        participant_ids = participant_ids[:max_subjects]
    cohort_mask = generate_trial_mask(data, trial_query) & np.isin(subjects, participant_ids)
    selected_ids = participant_ids[subject_indices] if subject_indices else participant_ids
    trial_mask = cohort_mask & np.isin(subjects, selected_ids)
    selected_rows = np.flatnonzero(trial_mask).tolist()
    if not selected_rows:
        raise ValueError("The trial selection contains no observations.")

## Fit the selected trials

The fixed parameters enter the model at their specified values. The optimizer searches the bounds in `parameters["free"]` to minimize the chosen loss. Inspection uses the parameter values already in the record.

In [7]:
#| code-summary: "Read vocabulary features for models that use them"

def load_features(spec):
    semantic = None
    if spec.get("embedding_path"):
        semantic = np.load(resolve_project_path(spec["embedding_path"], root))
    category = None
    if spec.get("emotion_feature_path"):
        values = np.load(resolve_project_path(spec["emotion_feature_path"], root))
        column = int(spec.get("feature_column", 6))
        category = values[:, column : column + 1]
    if spec.get("concat_features"):
        return np.concatenate([category, semantic], axis=1)
    return category if category is not None else semantic

In [8]:
#| code-summary: "Estimate parameters on the selected trials"

losses = {}
for name in models_to_fit:
    spec = model_configs[name]
    parameters = spec["parameters"]
    factory = model_factory_from_paths(spec["model_factory"], spec["components"])
    features = load_features(spec)
    loss_class = import_from_string(loss_fn_path)
    fitter_class = import_from_string(fit_alg_path)
    search = {**fit_hyperparams, "seed": spec["seed"], "bounds": parameters["free"]}
    fitter = fitter_class(data, features, parameters["fixed"], factory, loss_class, hyperparams=search)

    if pooled:
        result = fitter.fit(trial_mask, subject_id=-1)
    elif len(np.unique(subjects[trial_mask])) == 1:
        result = fitter.fit(trial_mask, subject_id=int(subjects[trial_mask][0]))
    else:
        result = fitter.fit_subjects(trial_mask)
    records[name] = result
    losses[name] = loss_class(factory, data, features)

In [9]:
#| code-summary: "Record the model and observations for each new fit"

for name in models_to_fit:
    spec = model_configs[name]
    record = records[name]
    path = resolve_project_path(data_path, root)
    configuration = {
        "model_factory": spec["model_factory"], "components": spec["components"],
        "data_path": path.relative_to(root).as_posix() if path.is_relative_to(root) else str(path),
        "loss_function": loss_fn_path, "fit_algorithm": fit_alg_path, "pooled": pooled,
        "selected_trial_indices": selected_rows, "cohort_trial_indices": np.flatnonzero(cohort_mask).tolist(),
        "feature_column": spec.get("feature_column", 6), "concat_features": spec.get("concat_features", False),
        "jax_enable_x64": enable_x64,
    }
    for key in ("embedding_path", "emotion_feature_path"):
        configuration[key] = spec.get(key, "")
        if configuration[key]:
            feature_path = resolve_project_path(configuration[key], root)
            configuration[key] = feature_path.relative_to(root).as_posix() if feature_path.is_relative_to(root) else str(feature_path)
    record.update(configuration, name=fit_paths[name].stem, model=name, label=spec.get("label", name),
                  data_tag=tag, run_tag=run_tag, trial_query=trial_query, data_query=trial_query,
                  simulation_function=spec.get("simulation_function", "jaxcmr.simulation.simulate_study_free_recall_and_forced_stop"),
                  jaxcmr_version=jaxcmr.__version__, jax_version=jax.__version__)
    record["fit_configuration"] = dict(configuration, parameters=spec["parameters"],
                                       search_settings={**fit_hyperparams, "seed": spec["seed"], "bounds": spec["parameters"]["free"]})

## Inspect the fitted parameters

A pooled fit gives one row for all selected participants. An individual fit gives one row per participant. For saved fits, the recorded objective comes from the original run. Rescoring evaluates the same parameters on the observation file available to this run.

In [10]:
#| code-summary: "Score each parameter row on its observations"

current_scores = {}
for name, record in records.items():
    data = datasets[observation_paths[name]]
    subjects = np.asarray(data["subject"]).ravel()
    if "selected_trial_indices" in record:
        rows = np.asarray(record["selected_trial_indices"], dtype=int)
    else:
        rows = np.flatnonzero(generate_trial_mask(data, record.get("trial_query", record.get("data_query"))))

    if name not in losses:
        factory = model_factory_from_paths(record["model_factory"], record["components"])
        loss_path = record["loss_function"].replace(
            "jaxcmr.loss.transform_sequence_likelihood.", "jaxcmr.loss.sequence_likelihood."
        )
        losses[name] = import_from_string(loss_path)(factory, data, load_features(record))
    fixed = record.get("fit_configuration", {}).get("parameters", {}).get("fixed", record["fixed"])
    free_names = tuple(record["free"])
    scores = []
    for index, participant in enumerate(record["fits"]["subject"]):
        fit_rows = rows if participant == -1 else rows[subjects[rows] == participant]
        evaluate = make_scipy_loss_fn(losses[name], jnp.asarray(fit_rows), fixed, free_names)
        values = jnp.asarray([record["fits"][key][index] for key in free_names])
        scores.append(float(evaluate(values)))
    current_scores[name] = scores

In [11]:
#| code-summary: "Check and save new fit records"

for name in models_to_fit:
    record = records[name]
    expected_ids = [-1] if pooled else np.unique(np.asarray(datasets[data_path]["subject"]).ravel()[selected_rows]).tolist()
    if sorted(record["fits"]["subject"]) != expected_ids:
        raise ValueError(f"{name}: fitted rows do not match selected participants.")
    if not np.allclose(record["fitness"], current_scores[name], rtol=1e-6, atol=1e-5):
        raise ValueError(f"{name}: fitted and recalculated objectives disagree.")
    record["recomputed_objective"] = current_scores[name]
    if pooled and loss_fn_path.endswith("ExcludeTerminationLikelihoodLoss"):
        record["recomputed_nll"] = current_scores[name][0]
    path = fit_paths[name]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(record, indent=2) + "\n")

Pass the fit JSON to [Simulating from saved fits](https://githubpsyche.github.io/jaxcmr/templates/simulation.html) to generate responses. The [example fit notes](https://github.com/githubpsyche/jaxcmr/blob/main/data/fits/README.md) explain the difference between the bundled fit’s recorded and current scores.

In [12]:
#| code-summary: "Show parameter estimates, scores and fit paths"

for name, path in fit_paths.items():
    record = records[name]
    participants = ["Pooled" if value == -1 else value for value in record["fits"]["subject"]]
    index = pd.Index(participants, name="Participant")
    estimates = pd.DataFrame({key: record["fits"][key] for key in record["free"]}, index=index)
    if name in models_to_fit:
        scores = {"Objective": record["fitness"]}
    else:
        scores = {"Recorded objective": record["fitness"], "Current objective": current_scores[name]}

    print(record.get("label", name))
    display(pd.DataFrame(scores, index=index))
    display(estimates.T.rename_axis("Parameter") if len(estimates) == 1 else estimates)
    print("Fit file:", path.relative_to(root) if path.is_relative_to(root) else path)

CMR


,Recorded objective,Current objective
Participant,,
Pooled,7202.029122,7202.101649


Participant,Pooled
Parameter,
encoding_drift_rate,0.681201
start_drift_rate,0.859833
recall_drift_rate,0.718986
learning_rate,0.364619
shared_support,22.650989
item_support,22.958860
primacy_scale,0.084171
primacy_decay,14.995865
choice_sensitivity,92.971052


Fit file: data/fits/Dupertuys2026_CMR_demo.json
